In [1]:
import torch
import torchvision.models as models
import pandas as pd
import time
import os


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [3]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(path):
    return round(os.path.getsize(path) / (1024 * 1024), 2)

def benchmark_inference(model, input_tensor, runs=50):
    model.eval()
    with torch.no_grad():
        start = time.time()
        for _ in range(runs):
            _ = model(input_tensor)
        end = time.time()
    return round((end - start) / runs * 1000, 2)  # ms


In [4]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)


In [5]:
models_info = []

# -------- VGG16 --------
vgg = models.vgg16(pretrained=False)
vgg.classifier[6] = torch.nn.Linear(4096, 25)
vgg.load_state_dict(torch.load("vgg16_final.pth", map_location=device))
vgg = vgg.to(device)

models_info.append({
    "Model": "VGG16",
    "Params (M)": round(count_parameters(vgg) / 1e6, 2),
    "Size (MB)": model_size_mb("vgg16_final.pth"),
    "Inference (ms)": benchmark_inference(vgg, dummy_input)
})

# -------- ResNet50 --------
resnet = models.resnet50(pretrained=False)
resnet.fc = torch.nn.Linear(resnet.fc.in_features, 25)
resnet.load_state_dict(torch.load("resnet50_final.pth", map_location=device))
resnet = resnet.to(device)

models_info.append({
    "Model": "ResNet50",
    "Params (M)": round(count_parameters(resnet) / 1e6, 2),
    "Size (MB)": model_size_mb("resnet50_final.pth"),
    "Inference (ms)": benchmark_inference(resnet, dummy_input)
})

# -------- MobileNetV2 --------
mobile = models.mobilenet_v2(pretrained=False)
mobile.classifier[1] = torch.nn.Linear(mobile.classifier[1].in_features, 25)
mobile.load_state_dict(torch.load("mobilenetv2_best.pth", map_location=device))
mobile = mobile.to(device)

models_info.append({
    "Model": "MobileNetV2",
    "Params (M)": round(count_parameters(mobile) / 1e6, 2),
    "Size (MB)": model_size_mb("mobilenetv2_best.pth"),
    "Inference (ms)": benchmark_inference(mobile, dummy_input)
})

# -------- EfficientNetB0 --------
eff = models.efficientnet_b0(pretrained=False)
eff.classifier[1] = torch.nn.Linear(eff.classifier[1].in_features, 25)
eff.load_state_dict(torch.load("efficientnetb0_final.pth", map_location=device))
eff = eff.to(device)

models_info.append({
    "Model": "EfficientNetB0",
    "Params (M)": round(count_parameters(eff) / 1e6, 2),
    "Size (MB)": model_size_mb("efficientnetb0_final.pth"),
    "Inference (ms)": benchmark_inference(eff, dummy_input)
})


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [6]:
df = pd.DataFrame(models_info)
df


,Model,Params (M),Size (MB),Inference (ms)
0,VGG16,134.36,512.57,21.53
1,ResNet50,23.56,90.18,9.04
2,MobileNetV2,2.26,8.84,5.90
3,EfficientNetB0,4.04,15.70,11.46
